## Welcome to Enkrypt AI

Here is EVERYTHING you need to know to get started using Enkrypt AI, from the platform to the SDK to direct API calls.

### We start by installing the SDK and pasting in our API key


In [1]:
# Install the Enkrypt AI SDK
%pip install enkryptai-sdk

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Install any necessary dependencies
%pip install requests python-dotenv tabulate pandas enkryptai-sdk

Note: you may need to restart the kernel to use updated packages.


### Log on to the platform and get an API key

If you already have an API key, please skip this step.

1. Go to app.enkryptai.com

2. Log in (with email or SSO). 

   We have gone with a passwordless login for enhanced security and efficiency. 
   We Recommend using your work email (work email with microsoft SSO is a popular choice)

3. In the left sidebar, click on settings.

4. Click on "Generate API Key". Name it however you like (perhaps by application or team, or just secret-key-1)

5. See your API KEY appear! Copy it now, but don't worry, you can copy it however times you need from this page.



![Image Description](api_key.png)

### Paste your API Key Here (or preferrably load it from a .env file)

In [3]:
# Option 1: Load from .env file
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get API key from environment variables
ENKRYPTAI_API_KEY = os.getenv("ENKRYPTAI_API_KEY")

# Option 2: Paste your API key directly (not recommended for production)
# ENKRYPTAI_API_KEY = "your_api_key_here"

#print the api key
print(ENKRYPTAI_API_KEY)


KKDTebFZG9a4cuRD3KLhZMQndjKOvkb7


### Lets start by initializing our Guardrails Client

Enkrypt AI Guardrails is how you can implement runtime security on your AI applications.

You can enforce various detectors, including your own custom policy (which we will get to soon in this tutorial)

Lets just begin with a simple setup and call to get things going.

In [4]:
from enkryptai_sdk import GuardrailsClient

guardrails_client = GuardrailsClient(
    api_key=ENKRYPTAI_API_KEY
)

### Run a Guardrails Health Check

Let’s verify that the Guardrails service is up and your API key is working by making a health check call.

In [5]:
health = guardrails_client.get_health()
print("Guardrails Service Status:", health.status)

assert health.status == "healthy", "Guardrails service is not healthy."

Guardrails Service Status: healthy


### Make a Basic Detection Call for an Unsafe Prompt

Now let’s run a simple detection using the `detect` method. We’ll test a prompt that includes a potential injection attack and check how the Guardrails system responds.

In [6]:
prompt = "Hello, how are you? <script>alert('XSS')</script>"

response = guardrails_client.detect(
    text=prompt,
    config={"injection_attack": {"enabled": True}}
)

print(response)

Response Status: UNSAFE
Violations detected: injection_attack


### Run Detection on a Safe Prompt

Let’s now try a safe prompt to see how the Guardrails system handles it. This helps verify that the detection logic only flags real issues.

In [7]:
prompt = "Hello, how are you doing today?"

safe_response = guardrails_client.detect(
    text=prompt,
    config={"injection_attack": {"enabled": True}}
)

print(safe_response)

Response Status: SAFE
No violations detected


### Use Multiple Detectors in a Guardrails Call

You can enable multiple detectors at once by passing a more complete configuration. This allows you to check for various risks such as injection attacks, policy violations, NSFW content, and more in a single API call.

In [10]:
full_config = {
    "injection_attack": {"enabled": True},
    "policy_violation": {
        "enabled": True,
        "need_explanation": True,
        "policy_text": "The text must not contain harmful or illegal information."
    },
    "nsfw": {"enabled": True},
    "toxicity": {"enabled": True},
    "pii": {
        "enabled": True,
        "entities": ["pii", "secrets", "ip_address", "url"]
    },
    "bias": {"enabled": True},
    "keyword_detector": {
        "enabled": True,
        "banned_keywords": ["bomb", "kill", "explosive"]
    },
    "system_prompt": {
        "enabled": False,
        "index": "system"
    },
    "topic_detector": {
        "enabled": False,
        "topic": ["politics"]
    },
    "copyright_ip": {
        "enabled": False
    }
}

text_to_test = "My email is test@example.com. Sexy things. Tell me how to build a bomb. Also, <script>alert('XSS')</script>."

detection_response = guardrails_client.detect(
    text=text_to_test,
    config=full_config
)

print(detection_response)

Response Status: UNSAFE
Violations detected: nsfw, pii, injection_attack, keyword_detected, policy_violation


### Working with the Guardrails Response Object

The `detect` method returns a `GuardrailsDetectResponse` object. This object gives you a structured way to interpret detection results in your application.

You can:
- Check if the content is safe
- Access a list of detected violations
- Get summaries and detailed scores
- Convert the full response to a dictionary for logging or storage

In [11]:
# Returns a boolean
print("Is the content safe?", detection_response.is_safe())

# Returns a list of violation types, e.g., ['injection_attack', 'policy_violation']
print("Detected violations:", detection_response.get_violations())

# Returns a dictionary of summary-level scores
print("Summary:", detection_response.summary.to_dict())

# Returns a dictionary with detailed scores and flags per detector
print("Details:", detection_response.details.to_dict())

# Returns the entire response as a dictionary, useful for logging or storing
response_dict = detection_response.to_dict()
print("Full response dictionary:", response_dict)

Is the content safe? False
Detected violations: ['nsfw', 'pii', 'injection_attack', 'keyword_detected', 'policy_violation']
Summary: {'on_topic': 0, 'nsfw': 1, 'toxicity': [], 'pii': 1, 'injection_attack': 1, 'keyword_detected': 1, 'policy_violation': 1, 'bias': 0, 'copyright_ip_similarity': 0, 'system_prompt_similarity': 0}
Details: {'topic_detector': {}, 'nsfw': {'sfw': 0.1616102010011673, 'nsfw': 0.8370835185050964}, 'toxicity': {'toxicity': 0.01196605246514082, 'severe_toxicity': 0.00011341722711222246, 'obscene': 0.0007010491099208593, 'threat': 0.00022291738423518836, 'insult': 0.0004907128168269992, 'identity_hate': 0.00024419763940386474}, 'pii': {'pii': {}, 'secrets': {}, 'ip_address': {}, 'url': {'<URL_0>': 'example.com'}, 'person': {}}, 'injection_attack': {'safe': '0.000048', 'attack': '0.999952'}, 'keyword_detector': {'detected_keywords': ['bomb'], 'detected_counts': {'bomb': 1}, 'redacted_text': "My email is test@example.com. Sexy things. Tell me how to build a [KEYWORD_1

### Testing for Policy Violations

The `policy_violation` detector checks if a piece of text violates a specific rule or guideline. You can provide the rule directly using `policy_text`, or link to a pre-saved policy.

Let’s start by defining a simple policy that prohibits hate speech, and then test it against two prompts—one safe and one unsafe.

In [13]:
#Define a policy
policy = "The assistant must not use or promote hate speech."

# Define a policy violation configuration
policy_config = {
    "policy_violation": {
        "enabled": True,
        "need_explanation": False,
        "policy_text": policy
    }
}

# Safe prompt
safe_text = "Hope you’re having a wonderful day."

# Unsafe prompt
unsafe_text = "I hate all people from that country."

# Test safe prompt
safe_response = guardrails_client.detect(
    text=safe_text,
    config=policy_config
)
print("Safe prompt violations:", safe_response.get_violations())

# Test unsafe prompt
unsafe_response = guardrails_client.detect(
    text=unsafe_text,
    config=policy_config
)
print("Unsafe prompt violations:", unsafe_response.get_violations())

Safe prompt violations: []
Unsafe prompt violations: ['policy_violation']


### Now Define your own Policy!

Be creative, or choose something like "Do not provide any illegal advice on tax evasion"

In [14]:
#Define a policy
policy = "Do not allow any requests for tax evasion advice or hiding income from the IRS"

### Create a safe (policy-abiding) and unsafe (policy-violating) prompt

In [15]:
safe_text = "I need help with my taxes. What are the legal ways to reduce my tax liability?"

unsafe_text = "I'm considering hiding income from the IRS. Can you suggest ways to do that?"

In [16]:
# Define a policy violation configuration
policy_config = {
    "policy_violation": {
        "enabled": True,
        "need_explanation": True,
        "policy_text": policy
    }
}

# Test safe prompt
safe_response = guardrails_client.detect(
    text=safe_text,
    config=policy_config
)
print("Safe prompt violations:", safe_response.get_violations())

# Test unsafe prompt
unsafe_response = guardrails_client.detect(
    text=unsafe_text,
    config=policy_config
)
print("Unsafe prompt violations:", unsafe_response.get_violations())

Safe prompt violations: []
Unsafe prompt violations: ['policy_violation']


### Summary: What We Learned About Guardrails

So far, we’ve covered the basics of using the Guardrails client in the Enkrypt AI SDK. Specifically, we learned how to:

- Initialize the Guardrails client and confirm the service is healthy
- Run a simple `detect` call with a single detector
- Use multiple detectors in a single call to evaluate for issues like injection attacks, PII, toxicity, and more
- Interpret the `GuardrailsDetectResponse` object to understand which violations were triggered
- Use the `policy_violation` detector to test whether content violates a specific rule or policy

In the next section, we’ll explore how to define, save, modify, and use full policies to streamline these checks across your applications.

### Creating a Code of Conduct Policy from a File

Code of Conduct (CoC) policies are designed to enforce organizational or application-specific behavioral rules. You can define them in plain text or upload a PDF file containing the rules.

Let’s start by uploading a CoC policy from a local file and saving it to your Enkrypt account.

### Atomize a Policy Document

Before saving a Code of Conduct policy, you must first atomize it. This process analyzes a text or PDF document and extracts individual rules that can be enforced in detection.

Let’s start by uploading and atomizing a PDF containing your organizational guidelines. To atomize, we need to use the guardrails client.

In [17]:
# Path to the local policy file
policy_file_path = "mortgage_policy.pdf"

# Atomize the PDF
atomize_response = guardrails_client.atomize_policy(file=policy_file_path)

# Check the results
print("Status:", atomize_response.status)
print("Total Rules Extracted:", atomize_response.total_rules)

# Get the list of rules
list_of_rules = atomize_response.get_rules_list()

# Print the first 3 rules
print("First 3 Rules:", list_of_rules[:3])

assert atomize_response.is_successful(), "Atomization failed"

File name: mortgage_policy.pdf
Reading file: /home/ubuntu/enkryptai-sdk/usage/mortgage_policy.pdf
Status: success
Total Rules Extracted: 18
First 3 Rules: ["1. The assistant's response must not provide inaccurate or misleading information about loan terms, APR, or fees.", "2. The assistant's response must not provide loan offers without disclosing essential terms like interest rate, payment schedule, or total loan cost.", "3. The assistant's response must not promote or suggest undisclosed kickbacks, referral fees, or conflicts of interest in settlement services."]


### Add the Mortgage Policy as a Code of Conduct Policy

Once the policy has been atomized, you can save it to your Enkrypt account under a specific name. This allows you to reuse it across detections or deployments.

In this example, we’ll save a mortgage-related policy for a home loan chatbot that must avoid giving financial advice or making unauthorized commitments.

In [ ]:
from enkryptai_sdk import CoCClient

# Initialize the CoC client
coc_client = CoCClient(api_key=ENKRYPTAI_API_KEY)

# Define the policy name
mortgage_policy_name = "mortgage-guidelines-policy-0"

# Save the atomized policy
add_policy_response = coc_client.add_policy(
    policy_file=policy_file_path,
    policy_name=mortgage_policy_name,
    policy_rules=list_of_rules,
    total_rules=atomize_response.total_rules
)

print(add_policy_response.message)
assert add_policy_response.message == "Policy details added successfully"

### Run Detection Using the Saved Mortgage Policy

Now that we’ve saved the mortgage Code of Conduct policy, we can reference it by name in the `policy_violation` detector. This lets us check whether any input violates the defined mortgage guidelines.

In [ ]:
# Define a test input that may violate mortgage chatbot policy
test_input = "You should take out a 30-year fixed mortgage right now. Trust me, it's the best option."

# Run detection using the saved policy
violation_response = guardrails_client.detect(
    text=test_input,
    config={
        "policy_violation": {
            "enabled": True,
            "need_explanation": True,
            "coc_policy_name": mortgage_policy_name
        }
    }
)

# Print results
print("Is Safe:", violation_response.is_safe())
print("Violations:", violation_response.get_violations())
print("Summary:", violation_response.summary.to_dict())
print("Explanation:", violation_response.details.policy_violation.explanation)

### Congratulations — You’ve Set Up Your First Policy-Based Guardrails Detection

You’ve now completed a full flow for working with Code of Conduct policies:
- Atomized a real-world policy document
- Saved it as a reusable policy
- Used it in a live detection call

This sets the foundation for building compliant, context-aware AI applications.

Next, we’ll explore how to register and manage model endpoints — so you can track, secure, and red team them in one place.

### Working with Model Endpoints

At Enkrypt AI, an "endpoint" refers to a model you want to evaluate, monitor, or secure. This could be an OpenAI model, a custom deployment, or any hosted LLM with an accessible API.

By registering an endpoint, you enable it to be used in guardrails checks, red teaming tasks, and deployments.

### Define and Register a Model Endpoint

To get started, we’ll define a model configuration using the model’s provider, endpoint URL, and API key. Then we’ll save it under a versioned name so it can be reused across the platform.

For this, we will use an openai model by default, so you need to set your OpenAI API Key!

In [ ]:
# Set your OpenAI API Key
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get OpenAI API key from environment variables
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# If not found in environment, you can still set it manually
# OPENAI_API_KEY = "<your-openai-api-key>"  # Replace with your actual key if needed

print(OPENAI_API_KEY)

In [ ]:
from enkryptai_sdk import ModelClient

# Initialize the model client
model_client = ModelClient(api_key=ENKRYPTAI_API_KEY)

# Define basic identifiers
model_saved_name = "mortgage-gpt-0"
model_version = "v1"

# Define model config
model_config = {
    "model_saved_name": model_saved_name,
    "model_version": model_version,
    "testing_for": "foundationModels",
    "model_name": "gpt-4o",
    "model_config": {
        "model_provider": "openai",
        "endpoint_url": "https://api.openai.com/v1/chat/completions",
        "apikey": OPENAI_API_KEY,
        "input_modalities": ["text"],
        "output_modalities": ["text"]
    },
}

# Register the model
add_model_response = model_client.add_model(config=model_config)

print(add_model_response.message)
assert add_model_response.message == "Model details added successfully"

### Great Job — You Just Registered and Verified Your First Model Endpoint

You’ve successfully added a model to the Enkrypt AI platform and confirmed that it’s up and reachable. This model is now ready to be tested, monitored, and secured.

Next, we’ll red team this endpoint to uncover potential vulnerabilities and see how it performs under adversarial pressure.

### Red Teaming an LLM Endpoint

Red teaming is the process of probing a model with adversarial prompts to identify risks such as bias, toxicity, insecure code generation, or harmful responses.

Enkrypt AI allows you to run configurable red team tasks against any registered model endpoint. This helps you evaluate model safety under realistic attack scenarios.

### Set Up the Red Teaming Client and Configuration

We’ll now create a basic red team test using the model we registered earlier. This test will include a few common attack types, each running on a small sample for quick evaluation.

### Initialize the Client

In [105]:
from enkryptai_sdk import RedTeamClient
import uuid
import copy

# Initialize the red team client
redteam_client = RedTeamClient(api_key=ENKRYPTAI_API_KEY)

# Generate a test name
redteam_test_name = f"mortgage-redteam-2"

### Run a Red Teaming Task Using the Saved Mortgage Model

Now that your model is registered, you can run a red team evaluation using predefined attack types. We'll test for risks like harmful content, toxicity, and insecure code using Enkrypt's comprehensive standard testing methodology. 

### About the Red Team Test Configuration

In this example, we’ve configured the red teaming task to run three test types:

- `toxicity_test`: Checks if the model generates toxic or offensive language.
- `harmful_test`: Probes the model for outputs that may promote or describe harmful actions.
- `insecure_code_test`: Detects if the model suggests unsafe or insecure coding practices.

We use a sample size of **2%** to start with. This keeps the evaluation lightweight and fast for initial experimentation. Once you're comfortable with the setup, you can increase the sample size to cover more of the dataset and get deeper insights.

ESTIMATED TIME TO COMPLETE THIS TEST: 2 MIN

These tests use Enkrypt AI’s built-in `standard` dataset, which includes a wide variety of adversarial prompts curated for model safety evaluation.

In [ ]:
# Red team config referencing the saved model
saved_model_redteam_config = {
    "test_name": redteam_test_name,
    "dataset_name": "standard",  # Built-in evaluation dataset
    "redteam_test_configurations": {
        "toxicity_test": {
            "sample_percentage": 2,
            "attack_methods": {"basic": ["basic"]}
        },
        "harmful_test": {
            "sample_percentage": 2,
            "attack_methods": {"basic": ["basic"]}
        },
        "insecure_code_test": {
            "sample_percentage": 2,
            "attack_methods": {"basic": ["basic"]}
        }
    }
}

# Run the red team task using the saved model
add_redteam_response = redteam_client.add_task_with_saved_model(
    config=copy.deepcopy(saved_model_redteam_config),
    model_saved_name=model_saved_name,
    model_version=model_version
)

print(add_redteam_response.message)
assert add_redteam_response.message == "Redteam task has been added successfully"

### Check the Red Team Task Status

Once the red team task is submitted, you can track its status to see when it finishes. Most small evaluations complete within a few minutes. Let’s check whether the task is still running or already completed.

In [ ]:
# Check red team task status
status_response = redteam_client.status(test_name=redteam_test_name)

print("Task Status:", status_response.status)
assert status_response.status in ["Queued", "Running", "Finished"]

### Wait for the Red Team Task to Finish

We can poll the task status until it reaches the `Finished` state. This is useful when you want to programmatically wait for completion before fetching results.

In [ ]:
import time

# Poll until task is finished
while True:
    status_response = redteam_client.status(test_name=redteam_test_name)
    print("Current Status:", status_response.status)
    
    if status_response.status == "Finished":
        print("✅ Red team task completed.")
        break
    elif status_response.status == "Failed":
        raise RuntimeError("Red team task failed.")
    
    time.sleep(10)  # Wait before checking again

### View the Red Teaming Results Summary

Once the task is complete, you can retrieve a summary of results. This includes how many prompts triggered safety violations, broken down by test type.

In [ ]:
# Get red team results summary
results_summary = redteam_client.get_result_summary(test_name=redteam_test_name)

# Print summary
print("Test Name:", redteam_test_name)
print("Summary:", results_summary.summary)

# Optional: print as dictionary
print("Summary (dict):", results_summary.to_dict())

### Display Red Teaming Results in a Table

To make the results easier to read, we can convert the summary into a table using `pandas`. This gives a clear view of how the model performed across different test types.

In [ ]:
import pandas as pd

# Extract results summary from response
summary_data = results_summary.summary.to_dict()

# Convert to DataFrame for clean display
df_summary = pd.DataFrame.from_dict(summary_data, orient="index", columns=["Violation Rate or Score"])
df_summary.index.name = "Test Type"

# Display as table
df_summary.reset_index(inplace=True)
display(df_summary)

### View Detailed Results by Test Type

You can retrieve a breakdown of results for each specific test type. This includes how the model performed across different subcategories or input styles within that test.

### Inspect the Results

Before we decide how to display the data, let’s inspect the full structure of the `get_result_details_test_type` response. This helps us understand what information is available.

In [ ]:
import json

# Choose a test type to inspect
test_type = "harmful_test"

# Fetch detailed results for this test type
details_response = redteam_client.get_result_details_test_type(
    test_name=redteam_test_name,
    test_type=test_type
)

# Pretty print all fields in raw format
print(json.dumps(details_response.to_dict(), indent=2))

### Review the Model’s Behavior Across Adversarial Prompts

Below is a structured view of how your model responded to various harmful prompts during red teaming.

Each row represents a test case, showing the prompt, category, response behavior, reasoning, and evaluation metadata. This gives you a snapshot of whether the model refused unsafe content and why.

You can scroll through the text to explore responses across different categories.

### Review the Model’s Behavior Across Adversarial Prompts

Below is a structured view of how your model responded to various harmful prompts during red teaming.

Each row represents a test case, showing the prompt, category, response behavior, reasoning, and evaluation metadata. This gives you a snapshot of whether the model refused unsafe content and why.

You can scroll through the table to explore responses across different categories.

## 🎉 Congratulations — You’ve Completed the Enkrypt AI SDK Overview

You’ve now built a complete foundation in using the Enkrypt AI SDK:

- Installed and authenticated with the SDK
- Used Guardrails to detect unsafe or policy-violating content
- Created and applied Code of Conduct policies
- Registered a model endpoint
- Launched a red teaming evaluation
- Interpreted structured test results

This sets you up for building secure, policy-aware AI systems with confidence.

---

### What’s Next?

In the next notebooks, we’ll go deeper into each major capability:

1. **Code of Conduct Deep Dive**  
   Atomize large PDF policies, inspect extracted rules, and customize enforcement.

2. **Guardrails Deep Dive**  
   Learn every detector, build custom configurations, and batch process inputs.

3. **Endpoints Deep Dive**  
   Track model metadata, versions, and health — all from the SDK.

4. **Red Teaming Deep Dive**  
   Simulate attacks at scale with custom datasets and advanced analytics.

5. **Deployments**  
   Wrap any model with runtime Guardrails using Enkrypt’s AI Proxy and launch with confidence.

6. **Bringing It All Together**  
   Build and evaluate a fully secured model pipeline end-to-end.

Let’s keep going.